In [1]:
import os
import pandas as pd

# Change this to your dataset root
DATASET_ROOT = "D:\IoMT\Dataset\main"

def get_class_distribution(split_dir):
    """
    Count samples per CSV file inside split_dir (train or test).
    Returns a DataFrame with class name and number of samples.
    """
    rows = []
    for fname in os.listdir(split_dir):
        if fname.endswith(".csv"):
            class_name = os.path.splitext(fname)[0]  # filename without .csv
            fpath = os.path.join(split_dir, fname)
            try:
                # Load just enough rows to count quickly
                df = pd.read_csv(fpath)
                n_samples = len(df)
            except Exception as e:
                print(f"Error reading {fpath}: {e}")
                n_samples = 0
            rows.append({"class": class_name, "samples": n_samples})
    return pd.DataFrame(rows).sort_values("samples", ascending=False)

# Train split
train_dir = os.path.join(DATASET_ROOT, "train")
train_dist = get_class_distribution(train_dir)
print("Train distribution:")
print(train_dist)

# Test split
test_dir = os.path.join(DATASET_ROOT, "test")
test_dist = get_class_distribution(test_dir)
print("\nTest distribution:")
print(test_dist)

# Merge for global view
train_dist["split"] = "train"
test_dist["split"] = "test"
all_dist = pd.concat([train_dist, test_dist], ignore_index=True)

# Pivot for summary
summary = all_dist.pivot_table(values="samples", index="class", columns="split", fill_value=0)
summary["total"] = summary.sum(axis=1)

print("\nOverall distribution (train/test/total):")
print(summary)

# Save to CSV for later inspection
#summary.to_csv("class_distribution_summary.csv")

Train distribution:
                                 class  samples
28         TCP_IP-DDoS-UDP2_train.pcap   207295
29         TCP_IP-DDoS-UDP3_train.pcap   206604
30         TCP_IP-DDoS-UDP4_train.pcap   206343
27         TCP_IP-DDoS-UDP1_train.pcap   206170
31         TCP_IP-DDoS-UDP5_train.pcap   205507
34         TCP_IP-DDoS-UDP8_train.pcap   204105
25         TCP_IP-DDoS-TCP3_train.pcap   204075
20         TCP_IP-DDoS-SYN2_train.pcap   203669
23         TCP_IP-DDoS-TCP1_train.pcap   202311
32         TCP_IP-DDoS-UDP6_train.pcap   202247
21         TCP_IP-DDoS-SYN3_train.pcap   202023
26         TCP_IP-DDoS-TCP4_train.pcap   200998
19         TCP_IP-DDoS-SYN1_train.pcap   199390
33         TCP_IP-DDoS-UDP7_train.pcap   197685
24         TCP_IP-DDoS-TCP2_train.pcap   197081
22         TCP_IP-DDoS-SYN4_train.pcap   196880
15        TCP_IP-DDoS-ICMP5_train.pcap   195032
11        TCP_IP-DDoS-ICMP1_train.pcap   194938
12        TCP_IP-DDoS-ICMP2_train.pcap   194818
18        TCP_IP-DDo

In [2]:
import pandas as pd

# --- Paste your train and test distributions into DataFrames ---
# Assuming you already have train_dist and test_dist from earlier code
# with columns ["class", "samples"]

# Define mapping from raw class name to broad group
def map_group(class_name):
    c = class_name.lower()
    if "arp_spoofing" in c:
        return "Spoofing"
    elif any(x in c for x in ["ping_sweep", "vulscan", "os_scan", "port_scan"]):
        return "Recon"
    elif any(x in c for x in ["mqtt"]):
        return "MQTT"
    elif any(x in c for x in ["dos-tcp", "dos_icmp", "dos-syn", "dos-udp", "dos_tcp", "dos_icmp", "dos_syn", "dos_udp"]):
        return "DoS"
    elif any(x in c for x in ["ddos"]):
        return "DDoS"
    elif "benign" in c:
        return "Benign"
    else:
        return "Other"

# Apply mapping
train_dist["group"] = train_dist["class"].apply(map_group)
test_dist["group"] = test_dist["class"].apply(map_group)

# Summarize
train_grouped = train_dist.groupby("group")["samples"].sum().reset_index().sort_values("samples", ascending=False)
test_grouped = test_dist.groupby("group")["samples"].sum().reset_index().sort_values("samples", ascending=False)

# Merge
all_grouped = pd.merge(train_grouped, test_grouped, on="group", how="outer", suffixes=("_train", "_test")).fillna(0)
all_grouped["total"] = all_grouped["samples_train"] + all_grouped["samples_test"]

print("Grouped distribution (train/test/total):")
print(all_grouped)

# Save for later
all_grouped.to_csv("grouped_class_distribution.csv", index=False)


Grouped distribution (train/test/total):
      group  samples_train  samples_test    total
0    Benign          19291         37607    56898
1      DDoS        1537476        349699  1887175
2       DoS        4631620       1035309  5666929
3      MQTT         262938        621013   883951
4     Other         416292         98432   514724
5     Recon         103726         27676   131402
6  Spoofing          16047          5868    21915


In [6]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Paths
DATASET_ROOT = r"D:\IoMT\Dataset\main"
train_dir = os.path.join(DATASET_ROOT, "train")
test_dir = os.path.join(DATASET_ROOT, "test")

# Define grouping rules
def map_group(class_name):
    c = class_name.lower()
    if "benign" in c:
        return "Benign"
    elif "arp_spoofing" in c:
        return "Spoofing"
    elif any(x in c for x in ["ping_sweep", "vulscan", "os_scan", "port_scan"]):
        return "Recon"
    elif "mqtt" in c:
        return "MQTT"
    elif "ddos" in c:
        return "DDoS"
    elif any(x in c for x in ["dos-tcp", "dos_icmp", "dos-syn", "dos-udp",
                              "dos_tcp", "dos_icmp", "dos_syn", "dos_udp"]):
        return "DoS"
    else:
        return "Other"

def load_and_group(input_dir):
    all_dfs = []
    for fname in os.listdir(input_dir):
        if fname.endswith(".csv"):
            class_name = os.path.splitext(fname)[0]
            group = map_group(class_name)

            fpath = os.path.join(input_dir, fname)
            try:
                df = pd.read_csv(fpath)
            except Exception as e:
                print(f"Error reading {fpath}: {e}")
                continue

            # Add label column
            df["label"] = group
            all_dfs.append(df)

    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)
    else:
        return pd.DataFrame()

# Load train and test folders
train_df = load_and_group(train_dir)
test_df = load_and_group(test_dir)

# Combine everything
full_df = pd.concat([train_df, test_df], ignore_index=True)
print(f"Full dataset shape: {full_df.shape}")
print(full_df["label"].value_counts())

# Split into 80/10/10 (stratified by label)
train_df, temp_df = train_test_split(
    full_df, test_size=0.2, stratify=full_df["label"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42
)

print(f"Train size: {len(train_df)}")
print(f"Val size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

# Save splits
output_dir = os.path.join(DATASET_ROOT, "final_split")
os.makedirs(output_dir, exist_ok=True)

train_df.to_csv(os.path.join(output_dir, "train.csv"), index=False)
val_df.to_csv(os.path.join(output_dir, "val.csv"), index=False)
test_df.to_csv(os.path.join(output_dir, "test.csv"), index=False)

print(f"Saved train/val/test CSVs to {output_dir}")



Full dataset shape: (9162994, 40)
label
DDoS        5846623
DoS         1707481
MQTT         883951
Other        514724
Recon        131402
Benign        56898
Spoofing      21915
Name: count, dtype: int64
Train size: 7330395
Val size: 916299
Test size: 916300
Saved train/val/test CSVs to D:\IoMT\Dataset\main\final_split


In [7]:
import os
import pandas as pd

# Path where you saved the final splits
DATASET_ROOT = r"D:\IoMT\Dataset\main\final_split"

# Function to check distribution
def check_distribution(file_path, split_name):
    df = pd.read_csv(file_path)
    counts = df["label"].value_counts().reset_index()
    counts.columns = ["class", "samples"]
    counts["split"] = split_name
    return counts

# Load distributions
train_dist = check_distribution(os.path.join(DATASET_ROOT, "train.csv"), "train")
val_dist   = check_distribution(os.path.join(DATASET_ROOT, "val.csv"), "val")
test_dist  = check_distribution(os.path.join(DATASET_ROOT, "test.csv"), "test")

# Combine
all_dist = pd.concat([train_dist, val_dist, test_dist], ignore_index=True)

# Pivot for summary
summary = all_dist.pivot_table(values="samples", index="class", columns="split", fill_value=0)
summary["total"] = summary.sum(axis=1)

print("Class distribution across splits:")
print(summary)

# Save for later inspection
#summary.to_csv(os.path.join(DATASET_ROOT, "class_distribution_summary.csv"))


Class distribution across splits:
split         test      train       val      total
class                                             
Benign      5690.0    45518.0    5690.0    56898.0
DDoS      584663.0  4677298.0  584662.0  5846623.0
DoS       170748.0  1365985.0  170748.0  1707481.0
MQTT       88395.0   707161.0   88395.0   883951.0
Other      51473.0   411779.0   51472.0   514724.0
Recon      13140.0   105122.0   13140.0   131402.0
Spoofing    2191.0    17532.0    2192.0    21915.0


In [13]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# Paths
DATASET_ROOT = r"D:\IoMT\Dataset\main\final_split"
train_file = os.path.join(DATASET_ROOT, "train.csv")
val_file   = os.path.join(DATASET_ROOT, "val.csv")
test_file  = os.path.join(DATASET_ROOT, "test.csv")

# Load datasets
train_df = pd.read_csv(train_file)
val_df   = pd.read_csv(val_file)
test_df  = pd.read_csv(test_file)

# Separate labels
y_train, y_val, y_test = train_df["label"], val_df["label"], test_df["label"]
X_train, X_val, X_test = train_df.drop(columns=["label"]), val_df.drop(columns=["label"]), test_df.drop(columns=["label"])

# --- Fix infinities & large values ---
for df in [X_train, X_val, X_test]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)  # convert inf → NaN

# --- Impute missing values (NaN, including those from inf) ---
imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train)
X_val   = imputer.transform(X_val)
X_test  = imputer.transform(X_test)

# --- Optionally clip extreme outliers (e.g., values > 1e10) ---
X_train = np.clip(X_train, -1e10, 1e10)
X_val   = np.clip(X_val, -1e10, 1e10)
X_test  = np.clip(X_test, -1e10, 1e10)

# --- Scale numeric features (z-score) ---
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# --- Encode labels ---
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_val   = label_encoder.transform(y_val)
y_test  = label_encoder.transform(y_test)

# Results
print("Shapes after preprocessing:")
print("Train:", X_train.shape, "Labels:", len(np.unique(y_train)))
print("Val:  ", X_val.shape, "Labels:", len(np.unique(y_val)))
print("Test: ", X_test.shape, "Labels:", len(np.unique(y_test)))
print("\nLabel mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))


Shapes after preprocessing:
Train: (7330395, 39) Labels: 7
Val:   (916299, 39) Labels: 7
Test:  (916300, 39) Labels: 7

Label mapping: {'Benign': np.int64(0), 'DDoS': np.int64(1), 'DoS': np.int64(2), 'MQTT': np.int64(3), 'Other': np.int64(4), 'Recon': np.int64(5), 'Spoofing': np.int64(6)}
